In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import random

# Step 1: Load and preprocess data
data = pd.DataFrame({
    'ProjectID': range(1, 11),
    'Probability': [0.5]*3 + [0.75]*5 + [0.1]*2,
    'ExpStartDate': pd.to_datetime([
        '2025-01-10', '2025-01-11', '2025-01-12',
        '2025-01-13', '2025-01-14', '2025-01-15',
        '2025-01-16', '2025-01-17', '2025-09-15', '2025-09-16'
    ]),
    'QA': [1]*5 + [0.5]*3 + [0.5]*2,
    'Mobile': [1]*8 + [0]*2,
    'Web': [1]*10,
    'DevOps': [1]*10,
    'UIUX': [1]*10,
    'PM': [1]*10
})

experts = ['QA', 'Mobile', 'Web', 'DevOps', 'UIUX', 'PM']
num_simulations = 10000

# Step 2: Define monthly buckets
data['Month'] = data['ExpStartDate'].dt.to_period('M')
months = sorted(data['Month'].unique())

# Step 3: Initialize simulation storage
simulation_results = {expert: {str(month): [] for month in months} for expert in experts}

# Step 4: Run simulation
for sim in range(num_simulations):
    for month in months:
        leads_in_month = data[data['Month'] == month]
        for expert in experts:
            total_hours = 0
            for _, lead in leads_in_month.iterrows():
                if random.random() < lead['Probability']:
                    total_hours += lead[expert]
            simulation_results[expert][str(month)].append(total_hours)

# Step 5: Aggregate results
# Step 5: Aggregate and prepare for CSV
rows = []
for expert in experts:
    for month in months:
        values = simulation_results[expert][str(month)]
        mean = np.mean(values)
        std = np.std(values)
        n = len(values)
        ci_margin = 1.96 * (std / np.sqrt(n))
        ci_lower = mean - ci_margin
        ci_upper = mean + ci_margin

        rows.append({
            'Month': str(month),
            'Expert': expert,
            'Mean': round(mean, 2),
            'Std Dev': round(std, 2),
            '95% CI Lower': round(ci_lower, 2),
            '95% CI Upper': round(ci_upper, 2)
        })

# Convert to DataFrame
results_df = pd.DataFrame(rows)

# Export to CSV
results_df.to_csv('expert_monthly_allocation.csv', index=False)

# Preview
print(results_df.head())


     Month  Expert  Mean  Std Dev  95% CI Lower  95% CI Upper
0  2025-01      QA  4.10     1.13          4.07          4.12
1  2025-09      QA  0.10     0.21          0.10          0.10
2  2025-01  Mobile  5.26     1.29          5.24          5.29
3  2025-09  Mobile  0.00     0.00          0.00          0.00
4  2025-01     Web  5.26     1.31          5.23          5.28


In [2]:
pd.read_json('expert_monthly_allocation.json')

,QA,Mobile,Web,DevOps,UIUX,PM
2025-01-01,"{'mean': 4.14, 'std_dev': 1.13, 'percentile_90...","{'mean': 5.25, 'std_dev': 1.3, 'percentile_90'...","{'mean': 5.26, 'std_dev': 1.3, 'percentile_90'...","{'mean': 5.25, 'std_dev': 1.31, 'percentile_90...","{'mean': 5.25, 'std_dev': 1.31, 'percentile_90...","{'mean': 5.24, 'std_dev': 1.3, 'percentile_90'..."
2025-09-01,"{'mean': 0.1, 'std_dev': 0.21, 'percentile_90'...","{'mean': 0.0, 'std_dev': 0.0, 'percentile_90':...","{'mean': 0.2, 'std_dev': 0.43, 'percentile_90'...","{'mean': 0.2, 'std_dev': 0.42, 'percentile_90'...","{'mean': 0.2, 'std_dev': 0.42, 'percentile_90'...","{'mean': 0.19, 'std_dev': 0.42, 'percentile_90..."
